# Kütüphaneler

In [1]:
import pandas as pd
import numpy as np
import pandas.io.formats.excel as pife
import warnings
import pickle
import os
import locale
from termcolor import colored
from pandas.errors import SettingWithCopyWarning
from datetime import datetime

# Ayarlar

In [2]:
os.system('color')
locale.setlocale(locale.LC_ALL, 'tr_TR')
pd.options.display.float_format = '{:,.2f}'.format
pife.ExcelFormatter.header_style = None

warnings.filterwarnings('ignore', category = UserWarning)
warnings.filterwarnings('ignore', category = pd.errors.ParserWarning)
warnings.filterwarnings('ignore', category = SettingWithCopyWarning)

base_path = 'C:\\Users\\105617\\Desktop\\Workspace\\Panel\\'

pickle_folder_path = base_path + 'pickle' + '\\'
data_folder_path = base_path + 'data' + '\\'
output_folder_path = base_path + 'output' + '\\'

# Yardımcı Fonksiyonlar

## Tablo Manipülasyonu

In [3]:
def add_h_space(df_original, space=1):
    df_final = df_original.copy()
    width = df_final.shape[1]
    for _ in range(space):
        df_final.loc[len(df_final)] = [np.nan] * width
    return df_final

def add_v_space(df_original, space=1):
    df_final = df_original.copy()
    df_null = pd.DataFrame([[np.nan] * space], columns=[np.nan] * space)
    df_final = pd.concat([df_final, df_null], axis=1)
    return df_final
    

def h_stack(df_list, space=3):
    df_first = df_list[0].copy().reset_index(drop=True)
    df_list_new = [df_first]
    df_null = pd.DataFrame({np.nan: [np.nan]})
    for df in df_list[1:]:
        df_temp = df.copy()
        for _ in range(space):
            df_list_new.append(df_null)
        df_list_new.append(df_temp.reset_index(drop=True))
    df_final = pd.concat(df_list_new, axis=1).reset_index(drop=True)
    return df_final


def v_stack(df_list, space=3):
    df_first = df_list[0].copy().reset_index(drop=True)
    df_list_new = [df_first]
    df_null = pd.DataFrame(columns=df_first.columns)
    df_null = add_h_space(df_null, space)
    for dft in df_list[1:]:
        df_temp = dft.copy()
        width_first = df_first.shape[1]
        width_temp = df_temp.shape[1]
        width_diff = width_first - width_temp

        if width_diff > 0:
            df_temp = add_v_space(df_temp, width_diff)
        elif width_diff < 0:
            df_first = add_v_space(df_first, -width_diff)

        df_new = pd.DataFrame([df_temp.columns], columns=df_first.columns)
        df_temp.columns = df_first.columns
        df_list_new.append(df_null)
        df_list_new.append(df_new)
        df_list_new.append(df_temp.reset_index(drop=True))

    df_final = pd.concat(df_list_new).reset_index(drop=True)
    return df_final


def insert_header(dfo, col):
    df = dfo.copy()
    header = [col] if type(col) is str else col
    return v_stack([pd.DataFrame(columns=header), df], 0)

def insert_corner(dfo):
    df = dfo.copy()
    return h_stack([pd.DataFrame(), v_stack([pd.DataFrame(), df], 0)], 1)

def insert_row(dfo, index=0, row=None, count=1):
    df = dfo.copy().reset_index(drop=True)
    if row is None:
        row = np.nan
    if type(row) is list and len(row) < df.shape[1]:
        row += [np.nan] * (df.shape[1] - len(row))
    if index < 0:
        index += df.shape[0]
    new_indices = [x for x in range(index)] + [x + count for x in range(index, len(df))]
    df.index = new_indices
    for i in range(count):
        df.loc[index + i] = row
    df = df.reset_index().sort_values('index').drop('index', axis=1).reset_index(drop=True)
    return df

## Verim Tabloları

In [4]:
def fix_verim_table(df, columns=None):
    start = 0
    if columns is None:
        start = 3
        columns = df.iloc[2, 1:]
    df = df.iloc[start:, 1:]
    df.columns = columns
    df.columns.name = None
    df = df.reset_index(drop=True)
    return df

def quick_export_verim_csv(data, output_file_name, sep=';', header=False, index=False, open_file=False):
    cl = [str(x) for x in range(0, 16)]
    df = pd.DataFrame()
    df['10'] = data
    for c in ['0', '15']:
        df[c] = 'x'
    for c in cl:
        if c not in ['0', '10', '15']:
            df[c] = np.nan
    df = df[cl]

    output_file_path = output_folder_path + output_file_name + '.csv'
    df.to_csv(output_file_path, sep=sep, header=header, index=index)

    if open_file:
        os.startfile(output_file_path)
    

## Hızlı Çıktı

In [5]:
def quick_export(df_export, output_file_name, sheet_name=None, open_file=True):
    output_file_path = output_folder_path + output_file_name + '.xlsx'
    writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')
    if type(df_export) is list:
        if sheet_name is None:
            sheet_name = [x + 1 for x in range(len(df_export))]
        for df, sheet in zip(df_export, sheet_name):
            df.to_excel(writer, sheet_name = sheet, index = False)
    else:
        if sheet_name is None:
            sheet_name = output_file_name
        df_export.to_excel(writer, sheet_name = sheet_name, index = False)
    writer.close()
    if open_file:
        os.startfile(output_file_path)


def quick_export_csv(df, file_name, open_file=False):
    df.to_csv(output_folder_path + file_name  + '.csv', sep=';', encoding='ANSI', index=False)

## Now

In [6]:
def now():
    # return datetime.strftime(datetime.now(), '%Y-%m-%d_%H-%M-%S')
    return datetime.strftime(datetime.now(), '%m%d%H%M')

## Pickle

In [7]:
def save_pickle(file, name, sub=''):
    os.makedirs(pickle_folder_path + sub, exist_ok = True)
    with open(pickle_folder_path + sub + name, 'wb') as handle:
        pickle.dump(file, handle, protocol=pickle.HIGHEST_PROTOCOL)

def load_pickle(name, sub=''):
    with open(pickle_folder_path + sub + name, 'rb') as handle:
        return pickle.load(handle)

# Listeler

In [8]:
segment_list, segment_title_list, segment_title_dict, bolum_list, filtered_bolum_list, sektor_list, filtered_sektor_list, sektor_title_dict, sektor_title_short_dict, sube_list, bolge_list = load_pickle('list_list')

# Validasyon

In [9]:
start_term, end_term = '2022.12', '2023.12'
file_path = data_folder_path + 'YPP - 2022.12.csv'

last_date = end_term
old_suffix = f'{start_term[2:4]}{start_term[-2:]}'
new_suffix = f'{end_term[2:4]}{end_term[-2:]}'

df_ms_all = load_pickle(f'df_ms_all_{old_suffix}_{new_suffix}')
# df_yz_all = load_pickle(f'df_yz_all_{old_suffix}_{new_suffix}')
# df_eds_all = load_pickle(f'df_eds_all_{old_suffix}_{new_suffix}')

sorted_date_list = load_pickle(f'sorted_date_list_v_{new_suffix}')
df_old = pd.read_csv(file_path, sep=';', encoding='ANSI')
df_yapi = pd.read_csv(data_folder_path + 'Yapılandırma.csv', sep=';')

for c in ['Nakdi', 'G.Nakdi', 'Toplam']:
    n = c + ' Risk (USD)'
    df_old[n] = df_old[n].apply(lambda x: int(x.replace(',', '')))

In [10]:
df_wide = []
mn = 'Müşteri No'
for c in df_yapi:
    n = f'Yapılandırma {c[-4:]}.{c[3:5]}'
    df = df_yapi[[c]].copy().dropna().drop_duplicates(keep='first').reset_index(drop=True).rename(columns={c:mn}).astype(int)
    df[n] = 1
    df_wide.append(df.copy())

## Max MS

In [26]:
def find_max_column(df, columns):
    values = [df[c] for c in columns if not np.isnan(df[c])]
    if len(values): return max(values)
    else: return np.nan
    
def find_first_occurrence(df, columns, threshold):
    value_list = [threshold if x >= threshold else x for x in df[columns].values]
    return  value_list.index(threshold) + 1 if threshold in value_list else np.nan

def get_df_pra_max(df_pra, df_wide, date_list, column='Müşteri Sınıfı', threhsolds=[2, 3], fill_na=True):
    c = column
    sorted_date_list = date_list
    mn = 'Müşteri No'
    a = 'Adı'

    df = df_pra.copy()
    si = sorted_date_list.index(start_term)
    ei = sorted_date_list.index(end_term) + 1
    dl = sorted_date_list[si:ei]

    if df_wide is not None:
        dfl = df_wide[si:ei]

        for dft, d in zip(dfl, dl):
            df = pd.merge(df, dft[['Müşteri No', column + ' ' + d]], on=mn, how='left')

    cs = [c + ' ' + d for d in dl]
    if fill_na is not None:
        df[cs] = df[cs].fillna(fill_na)
    cs = cs[1:]
    df[c + ' Max'] = df.apply(find_max_column, columns=cs, axis=1)
    for th in threhsolds:
        df[f'{column} {th} Geçiş Ayı'] = df.apply(find_first_occurrence, columns=cs, threshold=th, axis=1)

    return df.copy()

## Geçiş Özet

In [12]:
def create_pra_pass(df_old_max, rows, row_list, row_new=None, column='Müşteri Sınıfı', thresholds=[2, 3], drop_na=True):
    r = rows
    r_list = row_list
    r_dict = row_new
    short = ''.join([x[0] for x in column.split()])
    mn = 'Müşteri No'
    nrt = 'Nakdi Risk (USD)'
    msm = column + ' Max'
    msl = column + ' ' + last_date
    factor = 1_000_000
    dfv_list = []
    

    for find_risk, x, s in zip([False, True], [mn, nrt], ['Adet', 'Nakdi Risk']):
        dfp = df_old_max.copy()
        if find_risk:
            dfp[nrt] /= factor

        t_list = ['Genel Toplam']
        df = pd.DataFrame()
        df[r] = r_list

        c = x
        n = s
        dft = dfp[[r, c]].copy()
        if find_risk:
            dft = dft.groupby(r).sum().reset_index().rename(columns={c: n})
        else:
            dft = dft.groupby(r).count().reset_index().rename(columns={c: n})
        df = pd.merge(df, dft, on=r, how='left')
        t = df[n].sum()
        t_list.append(t)

        df = add_v_space(df)
        t_list.append(np.nan)

        for ss in thresholds:
            c = x
            n = f'{short} {ss} Geçen {s}'
            ga = f'{column} {ss} Geçiş Ayı'
            dfa = dfp.loc[dfp[msm] >= ss, [r, c, ga]].copy()
            if find_risk:
                dft = dfa[[r, c]].groupby(r).sum().reset_index().rename(columns={c: n})
            else:
                dft = dfa[[r, c]].groupby(r).count().reset_index().rename(columns={c: n})
            df = pd.merge(df, dft, on=r, how='left')
            t = df[n].sum()
            t_list.append(t)

            p = f'{short} {ss} Geçiş %'
            df[p] = df[n] / df[s]
            t = df[n].sum() / df[s].sum()
            t_list.append(t)

            c = f'{column} {ss} Geçiş Ayı'
            n = f'Ortalama {short} {ss} Geçiş Ayı'
            t = dfa[ga].mean()
            t_list.append(t)
            dft = dfa[[r, ga]].groupby(r).mean().reset_index().rename(columns={c: n})
            df = pd.merge(df, dft, on=r, how='left')

            df = add_v_space(df)
            t_list.append(np.nan)

        if drop_na:
            df = df.dropna(subset=list(df.columns)[2:], how='all').reset_index(drop=True)

        if r_dict is not None:
            if type(r_dict) is list:
                r_dict = dict(zip(r_list, r_dict))
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        df = add_h_space(df)
        df.loc[len(df)] = t_list

        header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, header)
        footer_end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)
        df.loc[len(df)] = ['Riski kapanan firmalar dahildir'] + [np.nan] * (len(df.columns) - 1)
        
        dfv_list.append(df.copy())

    return insert_corner(v_stack(dfv_list))


## Değişim

### Alt

In [13]:
def create_pra_dist_alt(df_pra, rows, row_list, use_thresholds=False, reorder=False, rename_dict=None, factor=1_000_000, find_risk=False):
    n_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    yrc = 'Yüksek Riskli Portföy Oranı'
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    c = rows
    c_list = row_list
    dfp = df_pra.copy()
    x = nrt if find_risk else mn

    df = pd.DataFrame()
    if use_thresholds:
        ettl = c_list
        etl = [f'{et[0]} - {et[1]}' for et in ettl]
        df[c] = etl
    else:
        df[c] = c_list if c_list is not None else sorted(df_pra.loc[df_pra[c].notnull(), c].unique())
        
    t_list = ['Genel Toplam']

    for n in n_list:
        dft = dfp.copy()
        if find_risk:
            dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
        else:
            dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).count().reset_index().rename(columns={x: n})
        if use_thresholds:
            nx_list = []
            for et in ettl:
                nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
                nx_list.append(nx)
            df[n] = nx_list
        else:
            df = pd.merge(df, dft, on=c, how='outer')
        t_list.append(df[n].sum())

    n = 'Toplam'
    dft = dfp.copy()
    if find_risk:
        dft = dft[[x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
    else:
        dft = dft[[x, c]].groupby(c).count().reset_index().rename(columns={x: n})
    if use_thresholds:
        nx_list = []
        for et in ettl:
            nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
            nx_list.append(nx)
        df[n] = nx_list
    else:
        df = pd.merge(df, dft, on=c, how='outer')
    t_list.append(df[n].sum())

    df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
    if rename_dict is not None:
        df[c] = df[c].map(rename_dict)
    df = add_h_space(df)
    df.loc[len(df)] = t_list
    if find_risk:
        df.iloc[:,1:] = df.iloc[:,1:] / factor
    df[yrc] = df[['Yüksek', 'Çok Yüksek']].sum(axis=1) / df['Toplam']
    if reorder:
        df.iloc[:-1, :] = df.iloc[:-1, :].sort_values(yrc, ascending=False)


    return df[[c, yrc]]

### Değişim

In [14]:
def create_pra_dist_change(df_old, df_new, rows, row_list, row_new=None, row_name=None, sort=0, use_thresholds=False, drop_na=True, study_scope=None):
    dfn = df_new.copy()
    dfo = df_old.copy()

    yrpo = 'Yüksek Riskli Portföy Oranı'

    r = rows
    r_list = row_list
    r_dict = row_new
    
    dfv_list = []

    for find_risk in [False, True]:
        df = pd.DataFrame()
        if use_thresholds:
            df[r] = [f'{et[0]} - {et[1]}' for et in r_list] + ['Genel Toplam']
        else:
            df[r] = r_list + ['Genel Toplam']
        df = add_v_space(df)

        dft = create_pra_dist_alt(dfo, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, start_term]
        df = pd.merge(df, dft, on=r, how='left')
        dft = create_pra_dist_alt(dfn, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, end_term]
        df = pd.merge(df, dft, on=r, how='left')

        if drop_na:
            df = df.dropna(subset=list(df.columns)[2:], how='all').reset_index(drop=True)

        df['Değişim'] = df[end_term] - df[start_term]
        

        if r_dict is not None:
            if type(r_dict) is list:
                r_dict = dict(zip(r_list, r_dict))
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        if sort is not None:
            df.loc[df[end_term] == 0, 'Değişim'] = df.loc[df[end_term] == 0, start_term] * -100
            df.loc[(df[end_term] == 0) & (df[start_term] == 0), 'Değişim'] = -1_000_000
            if type(sort) is str:
                if sort.startswith('-'):
                    df[:-1] = df[:-1].sort_values(sort[1:], ascending=False).reset_index(drop=True)
                else:
                    df[:-1] = df[:-1].sort_values(sort).reset_index(drop=True)
            else:
                if sort > 0:
                    df[:-1] = df[:-1].sort_values('Değişim').reset_index(drop=True)
                elif sort < 0:
                    df[:-1] = df[:-1].sort_values('Değişim', ascending=False).reset_index(drop=True)
            df['Değişim'] = df[end_term] - df[start_term]

        df = insert_row(df, -1)

        if row_name is not None:
            df = df.rename(columns={r: row_name})

        header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, header)
        df = insert_row(df, 0, [np.nan, np.nan, yrpo])
        footer_end = 'Reeskontlu' if find_risk else 'Adet'
        footer = f'{start_term} - {end_term} KR202, {footer_end}'
        df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)
        
        if study_scope is not None:
            footer_scope = f'{start_term} ve {end_term} çalışma kümelerine {study_scope[0]} ve üzeri firması giren {study_scope[1]} dikkate alınmıştır'
            df.loc[len(df)] = [footer_scope] + [np.nan] * (len(df.columns) - 1)
            df = add_h_space(df)

        dfv_list.append(df.copy())

    return insert_corner(v_stack(dfv_list))

In [15]:
def create_pra_dist_change_alt(df_old, df_new, rows, row_list, row_new=None, find_risk=False, sort=0, use_thresholds=False, drop_na=True, study_scope=None):
    dfn = df_new.copy()
    dfo = df_old.copy()

    r = rows
    r_list = row_list
    r_dict = row_new
    
    df = pd.DataFrame()
    if use_thresholds:
        df[r] = [f'{et[0]} - {et[1]}' for et in r_list] + ['Genel Toplam']
    else:
        df[r] = r_list + ['Genel Toplam']

    dft = create_pra_dist_alt(dfo, r, r_list, use_thresholds, find_risk=find_risk)
    dft.columns = [r, start_term]
    df = pd.merge(df, dft, on=r, how='left')
    dft = create_pra_dist_alt(dfn, r, r_list, use_thresholds, find_risk=find_risk)
    dft.columns = [r, end_term]
    df = pd.merge(df, dft, on=r, how='left')

    if drop_na:
        df = df.dropna(subset=list(df.columns)[2:], how='all').reset_index(drop=True)

    df['Değişim'] = df[end_term] - df[start_term]
    

    if r_dict is not None:
        if type(r_dict) is list:
            r_dict = dict(zip(r_list, r_dict))
        df[r] = df[r].apply(lambda x: r_dict.get(x, x))

    if sort is not None:
        df.loc[df[end_term] == 0, 'Değişim'] = df.loc[df[end_term] == 0, start_term] * -100
        df.loc[(df[end_term] == 0) & (df[start_term] == 0), 'Değişim'] = -1_000_000
        if type(sort) is str:
            if sort.startswith('-'):
                df[:-1] = df[:-1].sort_values(sort[1:], ascending=False)
            else:
                df[:-1] = df[:-1].sort_values(sort)
        else:
            if sort > 0:
                df[:-1] = df[:-1].sort_values('Değişim')
            elif sort < 0:
                df[:-1] = df[:-1].sort_values('Değişim', ascending=False)
        df['Değişim'] = df[end_term] - df[start_term]

    return df.copy()

### Scope

In [16]:
def create_pra_dist_change_scope(df_old, df_new, rows, row_list, row_new=None, use_thresholds=False, scope=20, study_scope=(30, 'şubeler')):
    dfn = df_new.copy()
    dfo = df_old.copy()

    yrpo = 'Yüksek Riskli Portföy Oranı'
    r = rows
    r_list = row_list
    r_dict = row_new

    dfv_list = []

    for find_risk in [False, True]:
        df = pd.DataFrame()
        if use_thresholds:
            df[r] = [f'{et[0]} - {et[1]}' for et in r_list] + ['Genel Toplam']
        else:
            df[r] = r_list + ['Genel Toplam']
        df = add_v_space(df)

        dft = create_pra_dist_alt(dfo, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, start_term]
        df = pd.merge(df, dft, on=r, how='left')
        dft = create_pra_dist_alt(dfn, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, end_term]
        df = pd.merge(df, dft, on=r, how='left')
        df['Değişim'] = df[end_term] - df[start_term]

        if r_dict is not None:
            if type(r_dict) is list:
                r_dict = dict(zip(r_list, r_dict))
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        df.loc[df[end_term] == 0, 'Değişim'] = df.loc[df[end_term] == 0, start_term] * -100
        df.loc[(df[end_term] == 0) & (df[start_term] == 0), 'Değişim'] = -1_000_000
        
        dfg = df.copy()
        dfg[:-1] = dfg[:-1].sort_values('Değişim')
        dfg = pd.concat([dfg[:scope], dfg[-1:]]).reset_index(drop=True)

        dfb = df.copy()
        dfb[:-1] = dfb[:-1].sort_values('Değişim', ascending=False)
        dfb = pd.concat([dfb[:scope], dfb[-1:]]).reset_index(drop=True)

        dfg['Değişim'] = dfg[end_term] - dfg[start_term]
        dfb['Değişim'] = dfb[end_term] - dfb[start_term]
        dfg = insert_row(dfg, -1)
        dfb = insert_row(dfb, -1)

        header_end = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        footer_end = 'Reeskontlu' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        header_good = f'İyileşen {scope}, {header_end}'
        header_bad = f'Kötüleşen {scope}, {header_end}'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        footer_scope = f'{start_term} ve {end_term} çalışma kümelerine {study_scope[0]} ve üzeri firması giren {study_scope[1]} dikkate alınmıştır'

        dfg = insert_header(dfg, header_good)
        dfg = insert_row(dfg, 0, [np.nan, np.nan, yrpo])
        dfg.loc[len(dfg)] = [footer] + [np.nan] * (len(dfg.columns) - 1)
        dfg.loc[len(dfg)] = [footer_scope] + [np.nan] * (len(dfg.columns) - 1)
        dfg = add_h_space(dfg)

        dfb = insert_header(dfb, header_bad)
        dfb = insert_row(dfb, 0, [np.nan, np.nan, yrpo])
        dfb.loc[len(dfb)] = [footer] + [np.nan] * (len(dfb.columns) - 1)
        dfb.loc[len(dfb)] = [footer_scope] + [np.nan] * (len(dfb.columns) - 1)
        dfb = add_h_space(dfb)

        dfv_list.append(h_stack([dfb, dfg]))

    return insert_corner(v_stack(dfv_list))

### Adet-Risk Değişimi

In [17]:
def create_pra_count_risk_change(df_old, df_new, rows, row_list, row_new=None, drop_na=True, sort=True, factor=1_000_000):
    dfn = df_new.copy()
    dfo = df_old.copy()

    dfn = dfn.loc[dfn['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])]
    dfo = dfo.loc[dfo['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])]
    r = rows
    r_list = row_list
    r_dict = row_new

    dfv_list = []

    for c, nn, pp, find_risk in zip(['Müşteri No', 'Nakdi Reeskontlu Toplam'], ['Adet', 'Nakdi Risk'], ['Adet Payı', 'Nakdi Risk Payı'], [False, True]):
        df = pd.DataFrame()
        df[r] = r_list
        df = add_v_space(df)
        t_list = ['Genel Toplam', np.nan]
        start = start_term + ' ' + pp
        end = end_term + ' ' + pp

        for dfp, d in zip([dfo, dfn], [start_term, end_term]):
            n = d + ' ' + nn
            p = d + ' ' + pp
            dft = dfp[[r, c]].copy()
            if find_risk:
                dft = dft.groupby(r).sum().reset_index().rename(columns={c: n})
                dft[n] /= factor
            else:
                dft = dft.groupby(r).count().reset_index().rename(columns={c: n})
            df = pd.merge(df, dft, on=r, how='outer')
            t = df[n].sum()
            t_list.append(t)
            df[p] = df[n] / t
            t_list.append(1)

        if drop_na:
            df = df.dropna(subset=df.columns[2:], how='all').reset_index(drop=True)

        if r_dict is not None:
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        if sort:
            df = df.sort_values([start, end, r], ascending=False).reset_index(drop=True)

        df.iloc[:, 2:] = df.iloc[:, 2:].fillna(0)

        df = add_h_space(df)
        df.loc[len(df)] = t_list

        df['Pay Değişimi'] = df[end] - df[start]
        df.iloc[-1, -1] = np.nan


        header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, 'Yüksek Riskli Portföy İçerisinde')
        df = insert_header(df, header)

        footer_end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)

        dfv_list.append(df.copy())

    return insert_corner(v_stack(dfv_list))

# Alt

In [18]:
def create_pra_pass_alt(df_old_max, rows, row_list, row_new=None, find_risk=False, last_date=last_date, drop_na=True):
    r = rows
    r_list = row_list
    r_dict = row_new
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    msm = 'Müşteri Sınıfı Max'
    msl = 'Müşteri Sınıfı ' + last_date
    factor = 1_000_000

    if find_risk:
        x = nrt
        s = 'Nakdi Risk'
    else:
        x = mn
        s = 'Adet'
    dfp = df_old_max.copy()
    if find_risk:
        dfp[nrt] /= factor

    t_list = ['Genel Toplam']
    df = pd.DataFrame()
    df[r] = r_list

    c = x
    n = s
    dft = dfp[[r, c]].copy()
    if find_risk:
        dft = dft.groupby(r).sum().reset_index().rename(columns={c: n})
    else:
        dft = dft.groupby(r).count().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=r, how='left')
    t = df[n].sum()
    t_list.append(t)

    df = add_v_space(df)
    t_list.append(np.nan)

    for ss in [2, 3]:
        c = x
        n = f'{ss}. Sınıfa Geçen {s}'
        ga = f'{ss}. Sınıfa Geçiş Ayı'
        dfa = dfp.loc[dfp[msm] == ss, [r, c, ga]].copy()
        if find_risk:
            dft = dfa[[r, c]].groupby(r).sum().reset_index().rename(columns={c: n})
        else:
            dft = dfa[[r, c]].groupby(r).count().reset_index().rename(columns={c: n})
        df = pd.merge(df, dft, on=r, how='left')
        t = df[n].sum()
        t_list.append(t)

        p = f'{ss}. Sınıfa Geçiş %'
        df[p] = df[n] / df[s]
        t = df[n].sum() / df[s].sum()
        t_list.append(t)

        c = f'{ss}. Sınıfa Geçiş Ayı'
        n = f'Ortalama {ss}. Sınıfa Geçiş Ayı'
        t = dfa[ga].mean()
        t_list.append(t)
        dft = dfa[[r, ga]].groupby(r).mean().reset_index().rename(columns={c: n})
        df = pd.merge(df, dft, on=r, how='left')

        df = add_v_space(df)
        t_list.append(np.nan)

    if drop_na:
        df = df.dropna(subset=list(df.columns)[2:], how='all').reset_index(drop=True)

    if r_dict is not None:
        if type(r_dict) is list:
            r_dict = dict(zip(r_list, r_dict))
        df[r] = df[r].apply(lambda x: r_dict.get(x, x))

    df.loc[len(df)] = t_list
    df = df[[r] + [f'{ss}. Sınıfa Geçiş %' for ss in [2, 3]]]
    return df.copy()


In [19]:
def create_pra_change_pass(df_old, df_new, rows, row_list, row_new=None, row_name=None, sort=None, drop_na=True, limits=None, study_scope=None):
    dfo = df_old.copy()

    r = rows
    r_list = row_list + ['Genel Toplam']
    r_dict = row_new
    
    nrt = 'Nakdi Reeskontlu Toplam'
    nrp = f'{start_term} Nakdi Risk Payı'
    rk = 'Risk Kategorisi'
    yrl = ['Çok Yüksek', 'Yüksek']
    yrpo = 'Yüksek Riskli Portföy Oranı'
    c2, c3 = '2. Sınıfa Geçiş %', '3. Sınıfa Geçiş %'
    tck, yrp, dp = 'Tüm Çalışma Kümesi', 'Yüksek Riskli Portföy', 'Diğer Portföy'
    n_list = [tck, yrp, dp]

    dfv_list = []

    for find_risk in [False, True]:
        df = pd.DataFrame()
        df[r] = r_list
        df = add_v_space(df)

        dft = dfo[[r, nrt]].copy().groupby(r).sum().reset_index()
        nrtt = dft[nrt].sum()
        dft.loc[len(dft)] = ['Genel Toplam', nrtt]
        dft[nrp] = dft[nrt] / nrtt
        dft = dft[[r, nrp]]

        df = pd.merge(df, dft, on=r, how='left')
        df = add_v_space(df)
        
        if r_dict is not None:
            if type(r_dict) is list:
                r_dict = dict(zip(r_list, r_dict))
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))
    
        df_all = dfo.copy()
        df_yr = dfo.loc[dfo[rk].isin(yrl)].copy()
        df_nyr = dfo.loc[~dfo[rk].isin(yrl)].copy()

        for dft, n in zip([df_all, df_yr, df_nyr], n_list):
            dfp = create_pra_pass_alt(dft, rows, row_list, row_new, find_risk).fillna(0)
            dfp[n] = dfp[c2] + dfp[c3]
            dfp = dfp.drop([c2, c3], axis=1)
            df = pd.merge(df, dfp, on=r, how='left')

        df['Yüksek Riskli / Diğer (x)'] = df[yrp] / df[dp]

        if drop_na:
            df = df.dropna(subset=yrp, axis=0).reset_index(drop=True)
            df[dp] = df[dp].fillna(0)

        df = add_v_space(df)
        
        dfp = create_pra_dist_change_alt(df_old, df_new, rows, row_list, row_new, find_risk)
        df = pd.merge(df, dfp, on=r, how='left')

        if sort is not None:
            df.iloc[:-1, :] = df.iloc[:-1, :].sort_values(nrp, ascending=False).reset_index(drop=True)
            # sort_asc = True if sort == 1 else False
            # df.iloc[:-1, :] = df.iloc[:-1, :].sort_values([tck, yrp, dp], ascending=sort_asc).reset_index(drop=True)

            if limits is not None:
                df = pd.concat([df[limits[0]:limits[1]], df[-1:]], ignore_index=True)

        df = insert_row(df, -1)

        if row_name is not None:
            df = df.rename(columns={r: row_name})
            
        go = f'{start_term} - {end_term} İçerisinde\n2 veya 3. Sınıfa Geçiş Oranları'
        header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, header)

        footer_end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)
        df.loc[len(df)] = ['Riski kapanan firmalar dahildir'] + [np.nan] * (len(df.columns) - 1)

        if study_scope is not None:
            footer_scope = f'{start_term} ve {end_term} çalışma kümelerine {study_scope[0]} ve üzeri firması giren {study_scope[1]} dikkate alınmıştır'
            df.loc[len(df)] = [footer_scope] + [np.nan] * (len(df.columns) - 1)

        df = insert_row(df, 0, [np.nan, np.nan, np.nan, np.nan, go, np.nan, np.nan, np.nan, np.nan, yrpo])
        dfv_list.append(df.copy())

    return insert_corner(v_stack(dfv_list))

## Filtrelenmiş Liste

In [20]:
def get_scope_filtered_list(df_old, df_new, column, filter_value=None, filter_column=None, scope=30):
    dfo = df_old.copy()
    dfn = df_new.copy()
    c = column

    if filter_value is not None:
        x = c if filter_column is None else filter_column
        y = filter_value
        dfo = dfo.loc[dfo[x] == y]
        dfn = dfn.loc[dfn[x] == y]

    dfo = dfo[c].value_counts().reset_index().rename(columns={'count': 'old'})
    dfn = dfn[c].value_counts().reset_index().rename(columns={'count': 'new'})
    df = pd.merge(dfo, dfn, on=c, how='inner')
    filtered_list = list(df.loc[df.apply(lambda x: x['old'] >= scope and x['new'] >= scope, axis=1), c])
    filtered_list.sort(key=locale.strxfrm)

    return filtered_list

# Çalışma Kümeleri

In [21]:
df_old_max = df_old.copy()
df_old_max = get_df_pra_max(df_old_max, df_wide, sorted_date_list, 'Yapılandırma', [1], 0)
df_old_max = get_df_pra_max(df_old_max, df_ms_all, sorted_date_list, 'Müşteri Sınıfı', [2, 3])
# df_old_max = get_df_pra_max(df_old_max, df_yz_all, sorted_date_list, 'Yapay Zeka Risk Sınıfı', [2, 3, 4, 5])
# df_old_max = get_df_pra_max(df_old_max, df_eds_all, sorted_date_list, 'Entegre Derece Skor', [6, 8, 10])

risk_category_list = [
    'Düşük Riskli',
    'Yönetilebilir',
    'Yakından İzlenmeli',
    'Yüksek Riskli'
]

new_filtered_bolum_list = [
    'Kurumsal Krediler Tahsis Bölümü',
    'Ticari Krediler Tahsis Bölümü',
    'Proje Finansmanı Bölümü',
    'Krediler İzleme Bölümü'
]

# Analiz

## Geçişler

In [22]:
c = 'Müşteri Sınıfı'
s = 1
df = df_old_max.copy()
df = df.loc[df[c + ' ' + start_term] <= s]

c = 'Yapılandırma'
s = 0
th = [1]
df = df.loc[df[c + ' ' + start_term] <= s]


df_yapi_pass = create_pra_pass(df, 'Risk Kategorisi', risk_category_list, None, c, th, False)

In [23]:
quick_export(df_yapi_pass, 'ypp yapı2')

In [24]:
def merge_ms_yapi(df, d):
    m = 'Müşteri Sınıfı ' + d
    y = 'Yapılandırma ' + d
    if df[y] == 1 or df[m] >= 2:
        return 2
    else:
        return df[m]

# def get_ms_yapi_max(df):
#     m = df['Müşteri Sınıfı Max']
#     y = df['Yapılandırma Max']
#     if pd.isnull(m):
#         m = 100
#     if pd.isnull(y):
#         y = 100
#     x = min(m, y)
#     if x == 100:
#         x = np.nan
#     return x

In [31]:
for d in sorted_date_list:
    df_old_max['MSY ' + d] = df_old_max.apply(merge_ms_yapi, d=d, axis=1)

df_old_max = get_df_pra_max(df_old_max, None, sorted_date_list, 'MSY', [2], 0)

In [43]:
c = 'MSY'
s = 1
th = [2]
df = df_old_max.copy()
df = df.loc[df[c + ' ' + start_term] <= s]

df_msy_pass = create_pra_pass(df, 'Risk Kategorisi', risk_category_list, None, c, th, False)

In [44]:
quick_export(df_msy_pass, 'ypp yapı3')

In [54]:
df = df_old_max.copy()
d = '2023.06'
df.loc[df['Yapılandırma ' + d] == 1, 'Müşteri Sınıfı ' + d].value_counts()

Müşteri Sınıfı 2023.06
2.0    3
Name: count, dtype: int64

In [56]:
c = 'Müşteri Sınıfı'
s = 1
th = [2, 3]
df = df_old_max.copy()
df = df.loc[df[c + ' ' + start_term] <= s]

df_ms_pass = create_pra_pass(df, 'Risk Kategorisi', risk_category_list, None, c, th, False)

In [ ]:
c = 'Yapay Zeka Risk Sınıfı'
s = 1
th = [2, 3, 4, 5]
df = df_old_max.copy()
df = df.loc[df[c + ' ' + start_term] <= s]

df_yz_pass = create_pra_pass(df, 'Risk Kategorisi', risk_category_list, None, c, th, False)

In [ ]:
c = 'Entegre Derece Skor'
s = 5
th = [6, 8, 10]
df = df_old_max.copy()
df = df.loc[df[c + ' ' + start_term] <= s]

df_eds_pass = create_pra_pass(df, 'Risk Kategorisi', risk_category_list, None, c, th, False)

In [ ]:
c = 'Gecikme Gün'
s = 1
th = [2, 3]
df = df_old_max.copy()
df = df.loc[df[c + ' ' + start_term] <= s]

df_ms_pass = create_pra_pass(df, 'Risk Kategorisi', risk_category_list, None, c, th, False)

In [134]:
quick_export([df_ms_pass, df_yz_pass, df_eds_pass], 'YPP 2022.12 Validasyon v2', ['MS', 'YZRS', 'EDS'])

## Değişimler

In [ ]:
df_pra_segment_change = create_pra_dist_change(df_old, df_new, 'Değerlendirmeye Esas Segment', segment_list, segment_title_list, 'Segment')
df_pra_bolum_change = create_pra_dist_change(df_old_bf, df_new_bf, 'İlgili Bölüm', new_filtered_bolum_list, None, 'Bölüm')
df_pra_sube_change = create_pra_dist_change(df_old, df_new, 'Şube Türü', filtered_sube_turu_list)
df_pra_eds_change = create_pra_dist_change(df_old, df_new, 'Entegre Derece Skor', [(1, 5), (6, 7), (8, 12)], row_name='Entegre Derece / Skor', use_thresholds=True)

## Geçiş + Değişimler

In [ ]:
df_pra_segment_change_pass = create_pra_change_pass(df_pra_old_max, df_new, 'Değerlendirmeye Esas Segment', segment_list, segment_title_list, 'Segment')
df_pra_bolum_change_pass = create_pra_change_pass(df_pra_old_max_bf, df_new_bf, 'İlgili Bölüm', new_filtered_bolum_list, row_name='Bölüm')
df_pra_sube_change_pass = create_pra_change_pass(df_pra_old_max, df_new, 'Şube Türü', filtered_sube_turu_list)

In [ ]:
df_pra_bolge_change_pass = create_pra_change_pass(df_pra_old_max, df_new, 'Bölge Adı', bolge_list, None, 'Bölge Müdürlüğü', 1)

sektor_scope_list = get_scope_filtered_list(df_pra_old_max, df_new, 'Tahsis Sektör Adı', scope=100)
df_pra_sektor_change_pass = create_pra_change_pass(df_pra_old_max, df_new, 'Tahsis Sektör Adı', sektor_scope_list, sektor_title_dict, 'Sektör', 1, study_scope=(100, 'sektörler'))

In [ ]:
kurumsal_sube_scope_list = get_scope_filtered_list(df_pra_old_max, df_new, 'Şube Adı', 'Kurumsal', 'Şube Türü')
ticari_sube_scope_list = get_scope_filtered_list(df_pra_old_max, df_new, 'Şube Adı', 'Ticari', 'Şube Türü')
kurumsal_ticari_sube_scope_list = kurumsal_sube_scope_list + ticari_sube_scope_list

karma_sube_scope_list = get_scope_filtered_list(df_pra_old_max, df_new, 'Şube Adı', 'Karma', 'Şube Türü')

In [ ]:
df_pra_kurumsal_ticari_sube_change_pass = create_pra_change_pass(df_pra_old_max, df_new, 'Şube Adı', kurumsal_ticari_sube_scope_list, None, 'Kurumsal / Ticari Şube', -1)

df_pra_karma_sube_change_pass_1 = create_pra_change_pass(df_pra_old_max, df_new, 'Şube Adı', karma_sube_scope_list, None, 'Karma Şube', -1, limits=(0, 25))
df_pra_karma_sube_change_pass_2 = create_pra_change_pass(df_pra_old_max, df_new, 'Şube Adı', karma_sube_scope_list, None, 'Karma Şube', -1, limits=(25, 50))
df_pra_karma_sube_change_pass = h_stack([df_pra_karma_sube_change_pass_1, df_pra_karma_sube_change_pass_2], 2)

# Başlıklar

In [ ]:
sheet_dict = {
    'Modeller': df_pra_model,
    'Geçişler': df_pra_pass,
    'Segment': df_pra_segment_change,
    'Bölüm': df_pra_bolum_change,
    'Şube': df_pra_sube_change,
    'EDS': df_pra_eds_change,
    'Sektör+': df_pra_sektor_change_pass,
    'Segment+': df_pra_segment_change_pass,
    'Bölüm+': df_pra_bolum_change_pass,
    'Şube+': df_pra_sube_change_pass,
    'Bölge+': df_pra_bolge_change_pass,
    'KurTic+': df_pra_kurumsal_ticari_sube_change_pass,
    'Karma+': df_pra_karma_sube_change_pass,
}

eds = 'Entegre Derece / Skor'
yzrs = 'Yapay Zeka Risk Sınıfı'
yrpo = 'Yüksek Riskli Portföy Oranı'

sheet_dict_list = [sheet_dict]

sheet_list = [item for sublist in [list(x.keys()) for x in sheet_dict_list] for item in sublist]
df_list = [item for sublist in [list(x.values()) for x in sheet_dict_list] for item in sublist]

# Final XLSX

In [ ]:
output_file_name = f'PRA {pra_v} Val {start_term}-{end_term}'
open_file = True

color_long_multi_trend, color_multi_trend, color_trend, row_based = True, True, True, True
output_file_path = output_folder_path + output_file_name + '.xlsx'
writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')

workbook = writer.book

for sheet, df in zip(sheet_list, df_list):
    df.to_excel(writer, sheet_name = sheet, index = False)

    worksheet = writer.sheets[sheet]
    worksheet.hide_gridlines(2)

writer.close()

if open_file:
    os.startfile(output_file_path)